# Proyecto DS3021 — Parte I: Adquisición de datos de Mercado Libre Perú

Notebook único del primer avance. La **sección 1, Extracción**, obtiene las publicaciones del sitio y
guarda el CSV crudo; la **sección 2, Limpieza**, lo convierte en el dataset final, en las tablas de control
de calidad y en la sección 4 del informe. Cada sección puede leerse por separado, pero comparten la
configuración de la celda 1.1.

## Cómo ejecutar

1. `pip install -r requirements.txt` (o ejecutar la celda de instalación de 1.1).
2. *Ejecutar todas las celdas*. Con la configuración de fábrica (`EJECUTAR_SCRAPER = False`) el notebook
   **no vuelve a extraer**: verifica `robots.txt`, diagnostica el CSV crudo entregado y rehace toda la
   limpieza en segundos, reproduciendo byte a byte los archivos versionados en el repositorio.
3. Para rehacer la captura desde cero, poner `EJECUTAR_SCRAPER = True` en 1.1. Requiere Chrome, toma
   alrededor de una hora por las pausas entre peticiones, puede pedir una verificación manual en la
   ventana del navegador y **sobrescribe el CSV crudo entregado** (`git checkout datos/crudos` lo devuelve).

## Qué produce

| Salida | Ruta |
| --- | --- |
| Dataset crudo | `datos/crudos/mercadolibre_raw.csv` |
| Dataset limpio | `datos/limpios/mercadolibre_limpio.csv` |
| Diagnóstico del crudo, calidad por columna, resumen por categoría | `datos/calidad/` |
| Filas separadas, duplicados retirados, precios excluidos | `datos/calidad/` |
| Evidencia de `robots.txt` | `datos/calidad/robots/` |
| Sección 4 del informe | `documentos/Dataset_preliminar.md` |

## Requisitos del enunciado y dónde se verifican

| Requisito | Dónde |
| --- | --- |
| Técnica de scraping (Selenium + BeautifulSoup) | 1.3 y 1.4 |
| Paginación | 1.4 (`avanzar_siguiente`) y 1.5 |
| ≥ 1 000 registros y ≥ 6 atributos | 1.7 y 2.5 |
| ≤ 10 % de nulos por columna | 2.5 |
| CSV en UTF-8 | 1.5 y 2.5 |
| `robots.txt`, `User-Agent`, `time.sleep()`, `try-except`, guardado incremental | 1.1, 1.2 y 1.5 |


# 1. Extracción

## 1.1 Configuración

Fuente, categorías, pausas, cabeceras, rutas de salida y columnas del CSV crudo. `EJECUTAR_SCRAPER`
decide si esta sección vuelve a visitar el sitio o solo trabaja sobre el CSV ya entregado.

In [1]:
# Ejecutar una sola vez en el entorno local (equivale a requirements.txt).
%pip install -q selenium beautifulsoup4 requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
from pathlib import Path
from urllib.robotparser import RobotFileParser
from urllib.parse import urlsplit, parse_qs
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# True vuelve a extraer del sitio y sobrescribe el CSV crudo entregado.
EJECUTAR_SCRAPER = False

CATEGORIAS = [
    {'nombre': 'celulares', 'url': 'https://listado.mercadolibre.com.pe/celulares'},
    {'nombre': 'laptops', 'url': 'https://listado.mercadolibre.com.pe/laptops'},
    {'nombre': 'televisores', 'url': 'https://listado.mercadolibre.com.pe/televisores'},
    {'nombre': 'refrigeradoras', 'url': 'https://listado.mercadolibre.com.pe/refrigeradoras'},
    {'nombre': 'audifonos', 'url': 'https://listado.mercadolibre.com.pe/audifonos'},
]

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/152.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'es-PE,es;q=0.9',
}

PAUSA_SEGUNDOS = 10
MAX_PAGINAS = 20

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CRUDOS = RAIZ / 'datos' / 'crudos'
LIMPIOS = RAIZ / 'datos' / 'limpios'
CALIDAD = RAIZ / 'datos' / 'calidad'
DOCUMENTOS = RAIZ / 'documentos'
for carpeta in (CRUDOS, LIMPIOS, CALIDAD, DOCUMENTOS, CALIDAD / 'robots'):
    carpeta.mkdir(parents=True, exist_ok=True)

RUTA_CSV = CRUDOS / 'mercadolibre_raw.csv'
RUTA_DIAGNOSTICO = CALIDAD / 'diagnostico_columnas.csv'
RUTA_DUPLICADOS = CALIDAD / 'duplicados_detectados.csv'

COLUMNAS = [
    'categoria', 'pagina', 'posicion_pagina', 'id_publicacion',
    'titulo', 'moneda', 'precio_actual_texto',
    'precio_anterior_texto', 'cuotas_texto', 'vendedor_texto',
    'url_publicacion', 'url_fuente', 'fecha_extraccion'
]

print('Raíz del proyecto:', RAIZ)
print('CSV crudo:', RUTA_CSV)
print('Categorías:', len(CATEGORIAS))
print('Máximo de páginas por categoría:', MAX_PAGINAS)
print('Pausa entre peticiones (s):', PAUSA_SEGUNDOS)
print('Volver a extraer del sitio:', EJECUTAR_SCRAPER)

Raíz del proyecto: c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre
CSV crudo: c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\datos\crudos\mercadolibre_raw.csv
Categorías: 5
Máximo de páginas por categoría: 20
Pausa entre peticiones (s): 10
Volver a extraer del sitio: False


## 1.2 Verificación de `robots.txt`

No basta con comprobar que el archivo responda: hay que leer sus reglas. La celda descarga el
`robots.txt` de los dos dominios involucrados —`www.mercadolibre.com.pe` y el subdominio de listados
`listado.mercadolibre.com.pe`, que tiene reglas propias—, los interpreta con `urllib.robotparser` y
consulta, para el `User-Agent` declarado, cada URL que el scraper va a visitar: la primera página de cada
categoría y una URL paginada con el sufijo `_Desde_N`.

Si alguna ruta estuviera prohibida, la celda falla y la extracción no debe ejecutarse. Los archivos
descargados se guardan en `datos/calidad/robots/` como evidencia fechada de la revisión.

In [3]:
URLS_ROBOTS = {
    'www': 'https://www.mercadolibre.com.pe/robots.txt',
    'listado': 'https://listado.mercadolibre.com.pe/robots.txt',
}

# Rutas que el scraper visita: portada de categoría y una página con el sufijo de paginación.
URLS_A_VERIFICAR = [c['url'] for c in CATEGORIAS] + [CATEGORIAS[0]['url'] + '_Desde_49']

ROBOTS_OK = False
try:
    lectores = {}
    for dominio, url in URLS_ROBOTS.items():
        respuesta = requests.get(url, headers=HEADERS, timeout=30)
        respuesta.raise_for_status()
        destino = CALIDAD / 'robots' / f'robots_{dominio}_{time.strftime("%Y-%m-%d")}.txt'
        destino.write_text(respuesta.text, encoding='utf-8')
        lector = RobotFileParser()
        lector.parse(respuesta.text.splitlines())
        lectores[dominio] = lector
        print(f'{url}: HTTP {respuesta.status_code}, evidencia en {destino.name}')

    permisos = {
        url: lectores['listado'].can_fetch(HEADERS['User-Agent'], url)
        for url in URLS_A_VERIFICAR
    }
    print()
    for url, permitido in permisos.items():
        print('PERMITIDO ' if permitido else 'BLOQUEADO ', url)

    # Rutas que el proyecto se compromete a no visitar; se comprueba que el archivo las restrinja.
    restringidas = ['https://www.mercadolibre.com.pe/gz/cart/',
                    'https://www.mercadolibre.com.pe/gz/checkout/',
                    'https://www.mercadolibre.com.pe/perfil/vendedor/']
    print()
    for url in restringidas:
        print('restringida por robots.txt:',
              not lectores['www'].can_fetch(HEADERS['User-Agent'], url), url)

    ROBOTS_OK = all(permisos.values())
    assert ROBOTS_OK, 'robots.txt no permite alguna de las rutas del scraper: no ejecutar la extracción.'
    print('\nVerificación aprobada: las rutas de listado están permitidas para el User-Agent declarado.')
except requests.RequestException as error:
    print('No se pudo consultar robots.txt:', error)
    print('Sin esta verificación no debe ejecutarse la extracción (sección 1.6).')

https://www.mercadolibre.com.pe/robots.txt: HTTP 200, evidencia en robots_www_2026-09-16.txt
https://listado.mercadolibre.com.pe/robots.txt: HTTP 200, evidencia en robots_listado_2026-09-16.txt

PERMITIDO  https://listado.mercadolibre.com.pe/celulares
PERMITIDO  https://listado.mercadolibre.com.pe/laptops
PERMITIDO  https://listado.mercadolibre.com.pe/televisores
PERMITIDO  https://listado.mercadolibre.com.pe/refrigeradoras
PERMITIDO  https://listado.mercadolibre.com.pe/audifonos
PERMITIDO  https://listado.mercadolibre.com.pe/celulares_Desde_49

restringida por robots.txt: True https://www.mercadolibre.com.pe/gz/cart/
restringida por robots.txt: True https://www.mercadolibre.com.pe/gz/checkout/
restringida por robots.txt: True https://www.mercadolibre.com.pe/perfil/vendedor/

Verificación aprobada: las rutas de listado están permitidas para el User-Agent declarado.


## 1.3 Selectores y lectura del HTML

BeautifulSoup sobre el HTML de cada página de resultados. Los nombres de clase del maquetado cambian con
frecuencia, así que están aislados en `buscar_clase` y en `detectar_tarjetas`: actualizar el scraper ante un
cambio del sitio significa tocar solo estas funciones. Se conservan los textos originales y los valores
ausentes; la conversión a número ocurre en la sección 2.

In [4]:
def buscar_clase(raiz, clases):
    for clase in clases:
        elemento = raiz.find(class_=clase)
        if elemento is not None:
            return elemento
    return None


def obtener_texto(elemento):
    if elemento is None:
        return None
    texto = elemento.get_text(' ', strip=True)
    return texto if texto != '' else None


def detectar_tarjetas(soup):
    tarjetas = soup.find_all('li', class_='ui-search-layout__item')
    if len(tarjetas) == 0:
        tarjetas = soup.find_all('div', class_='ui-search-result__wrapper')
    if len(tarjetas) == 0:
        tarjetas = soup.find_all('div', class_='poly-card')
    return tarjetas


def extraer_id(texto):
    texto = (texto or '').upper()
    posicion = texto.find('MPE')
    while posicion != -1:
        indice = posicion + 3
        while indice < len(texto) and texto[indice] in '-_':
            indice += 1
        inicio = indice
        while indice < len(texto) and texto[indice].isdigit():
            indice += 1
        if indice > inicio:
            return 'MPE' + texto[inicio:indice]
        posicion = texto.find('MPE', posicion + 3)
    return None


def extraer_precio(elemento):
    if elemento is None:
        return None
    moneda = buscar_clase(elemento, ['andes-money-amount__currency-symbol'])
    entero = buscar_clase(elemento, ['andes-money-amount__fraction'])
    centavos = buscar_clase(elemento, ['andes-money-amount__cents'])
    if entero is not None:
        resultado = obtener_texto(entero)
        if centavos is not None:
            resultado = resultado + ',' + obtener_texto(centavos)
        if moneda is not None:
            resultado = obtener_texto(moneda) + ' ' + resultado
        return resultado
    return obtener_texto(elemento)


def procesar_html(html, categoria, pagina, url_fuente):
    soup = BeautifulSoup(html, 'html.parser')
    tarjetas = detectar_tarjetas(soup)
    filas = []

    for posicion, tarjeta in enumerate(tarjetas, start=1):
        titulo = buscar_clase(
            tarjeta,
            ['poly-component__title', 'ui-search-item__title']
        )
        enlace = titulo if titulo is not None and titulo.name == 'a' else tarjeta.find('a')
        url_publicacion = enlace.get('href') if enlace is not None else None

        precio_actual = buscar_clase(
            tarjeta,
            ['poly-price__current', 'ui-search-price__second-line']
        )
        precio_anterior = buscar_clase(
            tarjeta,
            ['andes-money-amount--previous', 'ui-search-price__original-value']
        )
        moneda = buscar_clase(
            precio_actual if precio_actual is not None else tarjeta,
            ['andes-money-amount__currency-symbol']
        )
        cuotas = buscar_clase(
            tarjeta,
            ['poly-price__installments', 'ui-search-installments']
        )
        vendedor = buscar_clase(
            tarjeta,
            ['poly-component__seller', 'ui-search-official-store-label']
        )

        filas.append({
            'categoria': categoria,
            'pagina': pagina,
            'posicion_pagina': posicion,
            'id_publicacion': extraer_id(
                (url_publicacion or '') + ' ' + str(tarjeta)
            ),
            'titulo': obtener_texto(titulo),
            'moneda': obtener_texto(moneda),
            'precio_actual_texto': extraer_precio(precio_actual),
            'precio_anterior_texto': extraer_precio(precio_anterior),
            'cuotas_texto': obtener_texto(cuotas),
            'vendedor_texto': obtener_texto(vendedor),
            'url_publicacion': url_publicacion,
            'url_fuente': url_fuente,
            'fecha_extraccion': time.strftime('%Y-%m-%d %H:%M:%S'),
        })

    return filas

## 1.4 Navegación con Selenium

El sitio exige un navegador para servir las tarjetas de resultados, por lo que la navegación usa Selenium
—`requests` recibe páginas incompletas— y el análisis del HTML sigue a cargo de BeautifulSoup. La
paginación avanza con el botón real `Siguiente`, que produce las URL con el sufijo `_Desde_N`; si el sitio
pide una verificación, el notebook la delega a la persona que ejecuta en lugar de intentar evadirla.

In [5]:
def crear_driver():
    opciones = Options()
    opciones.add_argument('--start-maximized')
    opciones.add_argument('--lang=es-PE')
    opciones.add_argument('--user-agent=' + HEADERS['User-Agent'])
    return webdriver.Chrome(options=opciones)


def requiere_autenticacion(url):
    url = (url or '').lower()
    return (
        'account-verification' in url
        or '/login/' in url
        or '/lgz/' in url
    )


def esperar_tarjetas(driver):
    for intento in range(10):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        if len(detectar_tarjetas(soup)) > 0:
            return True
        if requiere_autenticacion(driver.current_url):
            return False
        time.sleep(2)
    return False


def revisar_autenticacion(driver, url_destino):
    if requiere_autenticacion(driver.current_url):
        print('\nACCIÓN MANUAL NECESARIA')
        print('Completa el inicio de sesión en la ventana de Chrome.')
        input('Cuando termine, vuelve al notebook y presiona ENTER: ')
        if requiere_autenticacion(driver.current_url):
            driver.get(url_destino)
        time.sleep(PAUSA_SEGUNDOS)


def avanzar_siguiente(driver):
    cuerpo = driver.find_element(By.TAG_NAME, 'body')
    for intento in range(3):
        cuerpo.send_keys(Keys.END)
        time.sleep(2)

    selectores = [
        'li.andes-pagination__button--next a',
        '.andes-pagination__button--next a',
        'a[title="Siguiente"]',
        'a[aria-label="Siguiente"]',
        'button[aria-label="Siguiente"]',
    ]

    botones = []
    for selector in selectores:
        botones.extend(driver.find_elements(By.CSS_SELECTOR, selector))

    if len(botones) == 0:
        for elemento in driver.find_elements(By.CSS_SELECTOR, 'a, button'):
            texto = (elemento.text or '').strip().lower()
            titulo = (elemento.get_attribute('title') or '').lower()
            etiqueta = (elemento.get_attribute('aria-label') or '').lower()
            if 'siguiente' in texto or 'siguiente' in titulo or 'siguiente' in etiqueta:
                botones.append(elemento)

    for boton in botones:
        try:
            url_anterior = driver.current_url
            html_anterior = driver.page_source
            boton.click()
            time.sleep(PAUSA_SEGUNDOS)
            revisar_autenticacion(driver, url_anterior)
            if driver.current_url != url_anterior or driver.page_source != html_anterior:
                return True
        except Exception:
            continue

    return False

## 1.5 Recorrido y guardado incremental

Recorre hasta veinte páginas por categoría. Cada página se añade al CSV apenas se procesa —guardado
incremental: una interrupción no pierde lo avanzado—, con `time.sleep()` entre peticiones y `try-except`
por página, de modo que una falla puntual no cancela la ejecución completa. El CSV se escribe en UTF-8 y
conserva nulos y duplicados: limpiarlos es tarea de la sección 2.

In [6]:
def ejecutar_scraper():
    driver = crear_driver()
    primera_pagina_guardada = True
    total = 0

    try:
        for categoria in CATEGORIAS:
            driver.get(categoria['url'])
            time.sleep(PAUSA_SEGUNDOS)
            revisar_autenticacion(driver, categoria['url'])
            paginas_vistas = set()

            for pagina in range(1, MAX_PAGINAS + 1):
                try:
                    if not esperar_tarjetas(driver):
                        revisar_autenticacion(driver, driver.current_url)

                    html = driver.page_source
                    url_actual = driver.current_url
                    filas = procesar_html(
                        html,
                        categoria['nombre'],
                        pagina,
                        url_actual
                    )

                    if len(filas) == 0:
                        print(f"[{categoria['nombre']}] página {pagina}: sin registros")
                        break

                    firma_pagina = tuple(
                        (fila['id_publicacion'], fila['titulo'], fila['url_publicacion'])
                        for fila in filas
                    )
                    if firma_pagina in paginas_vistas:
                        print(f"[{categoria['nombre']}] página repetida; fin de categoría")
                        break
                    paginas_vistas.add(firma_pagina)

                    df_pagina = pd.DataFrame(filas, columns=COLUMNAS)
                    modo_guardado = 'w' if primera_pagina_guardada else 'a'
                    df_pagina.to_csv(
                        RUTA_CSV,
                        mode=modo_guardado,
                        header=primera_pagina_guardada,
                        index=False,
                        encoding='utf-8-sig'
                    )
                    primera_pagina_guardada = False
                    total += len(df_pagina)

                    print(
                        f"[{categoria['nombre']}] página {pagina}: "
                        f"{len(df_pagina)} guardados | acumulado: {total}"
                    )
                    print('URL:', url_actual)

                    if pagina < MAX_PAGINAS:
                        if not avanzar_siguiente(driver):
                            print('No existe una página siguiente. Fin de categoría.')
                            break

                except Exception as error:
                    print(
                        f"Error en {categoria['nombre']}, página {pagina}: {error}"
                    )
                    break

    finally:
        driver.quit()

    if total == 0:
        raise RuntimeError('No se extrajeron registros.')

    print('Extracción terminada. Registros crudos:', total)
    return pd.read_csv(RUTA_CSV)

## 1.6 Ejecución

Con `EJECUTAR_SCRAPER = False` se carga el CSV crudo ya entregado, que es lo que necesita la sección 2.
Con `True` se vuelve a extraer del sitio, siempre que la verificación de `robots.txt` haya sido aprobada.

In [7]:
if EJECUTAR_SCRAPER:
    assert ROBOTS_OK, 'Verificar robots.txt (1.2) antes de extraer.'
    df_final = ejecutar_scraper()
else:
    df_final = pd.read_csv(RUTA_CSV, encoding='utf-8-sig')
    print('Extracción omitida (EJECUTAR_SCRAPER = False). Se usa el CSV crudo entregado:', RUTA_CSV)

print('Dimensiones:', df_final.shape)
display(df_final.head(10))

Extracción omitida (EJECUTAR_SCRAPER = False). Se usa el CSV crudo entregado: c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\datos\crudos\mercadolibre_raw.csv
Dimensiones: (5080, 13)


,categoria,pagina,posicion_pagina,id_publicacion,titulo,moneda,precio_actual_texto,precio_anterior_texto,cuotas_texto,vendedor_texto,url_publicacion,url_fuente,fecha_extraccion
0,celulares,1,1,MPE55308605,Apple iPhone 17 Pro Max (256 GB) - Color plata,S/,S/ 4.849,NaN,"6 cuotas de S/ 808 , 17 sin interés",NaN,https://www.mercadolibre.com.pe/apple-iphone-1...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
1,celulares,1,2,MPE65503965,Celular Samsung Galaxy S26 Ultra 512gb,S/,S/ 4.129,S/ 6.899,"6 cuotas de S/ 688 , 21 sin interés",NaN,https://www.mercadolibre.com.pe/celular-samsun...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
2,celulares,1,3,MPE55308632,Apple iPhone 17 Pro Max (256 GB) - Azul profundo,S/,S/ 4.849,NaN,"6 cuotas de S/ 808 , 17 sin interés",NaN,https://www.mercadolibre.com.pe/apple-iphone-1...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
3,celulares,1,4,MPE55308781,Apple iPhone 17 Pro (256 GB) - Azul profundo,S/,S/ 4.479,S/ 6.000,"6 cuotas de S/ 746 , 50 sin interés",NaN,https://www.mercadolibre.com.pe/apple-iphone-1...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
4,celulares,1,5,MPE47797906,Samsung Galaxy A56 5G Dual Sim 256GB 12GB Negr...,S/,S/ 1.517,S/ 2.199,"6 cuotas de S/ 252 , 88 sin interés",NaN,https://www.mercadolibre.com.pe/samsung-galaxy...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
5,celulares,1,6,MPE47797574,Cel Samsung A56 12 Gb Ram 256 Gb 5g 6.7 Pulgad...,S/,S/ 1.464,S/ 2.449,"6 cuotas de S/ 244 , 04 sin interés",NaN,https://www.mercadolibre.com.pe/cel-samsung-a5...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
6,celulares,1,7,MPE50678684,Motorola Edge 60 Fusion 8GB RAM 256GB ROM azul...,S/,"S/ 987,90",S/ 1.500,"6 cuotas de S/ 164 , 65 sin interés",NaN,https://www.mercadolibre.com.pe/motorola-edge-...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
7,celulares,1,8,MPE48954564,Samsung A56 5G 5G Dual SIM 256 GB gris 12 GB RAM,S/,S/ 1.464,S/ 2.449,"6 cuotas de S/ 244 , 04 sin interés",NaN,https://www.mercadolibre.com.pe/samsung-a56-5g...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
8,celulares,1,9,MPE45513937,Samsung Galaxy S25 Ultra 512gb Titanium Black,S/,S/ 3.339,S/ 6.999,"6 cuotas de S/ 556 , 54 sin interés",NaN,https://www.mercadolibre.com.pe/samsung-galaxy...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
9,celulares,1,10,MPE67588045,Xiaomi Pocophone Poco X8 Pro Negro 12+512 GB,S/,S/ 1.599,S/ 2.500,"6 cuotas de S/ 266 , 50 sin interés",NaN,https://www.mercadolibre.com.pe/xiaomi-pocopho...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05


## 1.7 Diagnóstico del dataset crudo

Reporta nulos y duplicados sin modificar el CSV crudo y comprueba los mínimos del enunciado. Las columnas
con muchos nulos se resuelven en la sección 2, no aquí.

In [8]:
diagnostico = pd.DataFrame({
    'columna': df_final.columns,
    'nulos': [df_final[columna].isnull().sum() for columna in df_final.columns],
    'porcentaje_nulos': [
        round(df_final[columna].isnull().mean() * 100, 2)
        for columna in df_final.columns
    ],
    'valores_unicos': [
        df_final[columna].nunique(dropna=True)
        for columna in df_final.columns
    ],
})

duplicados = df_final[
    df_final['id_publicacion'].notnull()
    & df_final.duplicated('id_publicacion', keep=False)
]

diagnostico.to_csv(RUTA_DIAGNOSTICO, index=False, encoding='utf-8-sig')
duplicados.to_csv(RUTA_DUPLICADOS, index=False, encoding='utf-8-sig')

print('Registros crudos:', len(df_final))
print('Columnas:', len(df_final.columns))
print('IDs únicos:', df_final['id_publicacion'].nunique(dropna=True))
print('Filas con ID repetido:', len(duplicados))
print('Cumple mínimo de 1000 registros:', len(df_final) >= 1000)
print('Cumple mínimo de 6 atributos:', len(df_final.columns) >= 6)
print('CSV crudo:', RUTA_CSV)
print('Diagnóstico:', RUTA_DIAGNOSTICO)

display(df_final['categoria'].value_counts(dropna=False))
display(diagnostico.sort_values('porcentaje_nulos', ascending=False))

Registros crudos: 5080
Columnas: 13
IDs únicos: 4542
Filas con ID repetido: 685
Cumple mínimo de 1000 registros: True
Cumple mínimo de 6 atributos: True
CSV crudo: c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\datos\crudos\mercadolibre_raw.csv
Diagnóstico: c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\datos\calidad\diagnostico_columnas.csv


categoria
televisores       1200
audifonos         1200
laptops           1040
celulares          960
refrigeradoras     680
Name: count, dtype: int64

,columna,nulos,porcentaje_nulos,valores_unicos
9,vendedor_texto,3255,64.07,107
7,precio_anterior_texto,2036,40.08,942
8,cuotas_texto,106,2.09,2337
0,categoria,0,0.00,5
1,pagina,0,0.00,20
2,posicion_pagina,0,0.00,60
3,id_publicacion,0,0.00,4542
4,titulo,0,0.00,4249
5,moneda,0,0.00,1
6,precio_actual_texto,0,0.00,1969


# 2. Limpieza

Toma el CSV crudo de la sección 1 y produce el dataset final. No vuelve a consultar Mercado Libre.
Conserva los valores ausentes: no inventa precios anteriores ni vendedores.

## 2.1 Carga

Todo se lee como texto para no perder ceros ni formatos antes de decidir cada conversión. `fila_csv`
guarda el número de línea del archivo original para poder rastrear cualquier registro.

In [9]:
raw = pd.read_csv(RUTA_CSV, encoding='utf-8-sig', dtype='string')
df = raw.copy()
df.insert(0, 'fila_csv', range(2, len(df) + 2))
print('Archivo:', RUTA_CSV)
print('Dimensiones originales:', raw.shape)
print(raw.isna().mean().mul(100).round(2).to_string())

Archivo: c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\datos\crudos\mercadolibre_raw.csv
Dimensiones originales: (5080, 13)
categoria                 0.00
pagina                    0.00
posicion_pagina           0.00
id_publicacion            0.00
titulo                    0.00
moneda                    0.00
precio_actual_texto       0.00
precio_anterior_texto    40.08
cuotas_texto              2.09
vendedor_texto           64.07
url_publicacion           0.00
url_fuente                0.00
fecha_extraccion          0.00


## 2.2 Normalización e identificadores
Los precios usan punto para miles y coma para decimales: `S/ 4.849` se convierte en `4849.00` y `S/ 987,90` en `987.90`.

El identificador original puede proceder de `/p/MPE...`, mientras la misma URL contiene otro código en `wid`. Para identificar la oferta se prioriza `wid` y, si falta, el código de una ruta directa que no contenga `/p/`. Se mantiene el identificador del scraper como `id_extraido`. Las filas sin un identificador de oferta recuperable se separan para revisión; no se equipara automáticamente un código de catálogo a una publicación. Esta regla se basa en la estructura del CSV y no implica una verificación en vivo de las ofertas.

In [10]:
for c in raw.columns:
    df[c] = df[c].str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)

def oferta_id(url):
    if pd.isna(url):
        return pd.NA
    u = urlsplit(url)
    parametros = parse_qs(u.query + '&' + u.fragment)
    wid = parametros.get('wid', [''])[0]
    if re.fullmatch(r'MPE[0-9]+', wid):
        return wid
    m = re.search(r'/MPE-?([0-9]+)(?:-|/|$)', u.path)
    if m and '/p/' not in u.path:
        return 'MPE' + m.group(1)
    return pd.NA

def precio(serie):
    s = serie.str.replace(r'^S/\s*', '', regex=True).str.replace(' ', '', regex=False)
    valido = s.str.fullmatch(r'(?:[0-9]{1,3}(?:\.[0-9]{3})+|[0-9]+)(?:,[0-9]{1,2})?', na=False)
    return pd.to_numeric(s.where(valido).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

df = df.rename(columns={'id_publicacion': 'id_extraido'})
df['id_publicacion'] = df['url_publicacion'].map(oferta_id).astype('string')
df['precio_actual'] = precio(df['precio_actual_texto'])
df['moneda'] = df['moneda'].replace({'S/': 'PEN'})
df['numero_cuotas'] = pd.to_numeric(df['cuotas_texto'].str.extract(r'^(\d+) cuotas? de ', expand=False), errors='coerce').astype('Int64')
importe = df['cuotas_texto'].str.extract(r'cuotas? de (S/ [\d., ]+)', expand=False).str.strip()
df['monto_cuota'] = precio(importe)
for c in ['pagina', 'posicion_pagina']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
df['fecha_extraccion'] = pd.to_datetime(df['fecha_extraccion'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
print('IDs originales diferentes del ID de oferta:', int((df.id_extraido != df.id_publicacion).fillna(False).sum()))
print('Filas sin ID de oferta recuperable:', int(df.id_publicacion.isna().sum()))
print('Precios actuales no interpretables:', int(df.precio_actual.isna().sum()))

IDs originales diferentes del ID de oferta: 2599
Filas sin ID de oferta recuperable: 56
Precios actuales no interpretables: 0


## 2.3 Calidad y duplicados
Se separan registros sin campos esenciales válidos. Para ofertas repetidas se conserva la observación más reciente y, en empates, la última fila del archivo. No se combinan atributos de observaciones distintas. El listado es una muestra de resultados de búsquedas; `categoria` representa la búsqueda de origen, que puede incluir accesorios u otros productos.

Las cuotas ausentes permanecen nulas. Un texto ausente no demuestra ausencia de financiamiento. Tampoco se asume que un precio anterior ausente sea igual al precio actual.

In [11]:
invalidas = (df['id_publicacion'].isna() | df['titulo'].isna() | df['categoria'].isna()
             | df['precio_actual'].isna() | df['precio_actual'].le(0)
             | ~df['moneda'].eq('PEN') | df['fecha_extraccion'].isna()
             | df['pagina'].isna() | df['posicion_pagina'].isna())
revision = df.loc[invalidas].copy()
revision.to_csv(CALIDAD / 'filas_para_revision.csv', index=False, encoding='utf-8')
validas = df.loc[~invalidas].sort_values(['fecha_extraccion', 'fila_csv'], kind='stable')
repetidas = validas.duplicated('id_publicacion', keep='last')
validas.loc[repetidas].to_csv(CALIDAD / 'duplicados_retirados.csv', index=False, encoding='utf-8')
unicas = validas.loc[~repetidas].sort_values('fila_csv').copy()
print('Filas para revisión:', len(revision))
print('Repeticiones retiradas:', int(repetidas.sum()))
print('Ofertas únicas antes de exclusiones:', len(unicas))

Filas para revisión: 56
Repeticiones retiradas: 605
Ofertas únicas antes de exclusiones: 4419


## 2.4 Precios cuestionados
Siete ofertas se excluyen por un precio incoherente con el producto anunciado, revisadas una por una y confirmadas en el CSV crudo: no provienen de la conversión numérica. Son exclusiones puntuales de esta muestra, no un umbral general: no se elimina ningún registro solo por ser caro o barato, y los demás valores extremos se conservan. La lista es explícita para que la decisión quede auditable; la evidencia se guarda en `precios_excluidos.csv`, sin verificar el precio real ni imputar reemplazos.

In [12]:
MOTIVO = ('Precio cuestionado por incoherencia con el producto; confirmado en el CSV crudo. '
          'Exclusión para análisis, sin verificación del precio real ni imputación.')
IDS_EXCLUIDOS = [
    'MPE824658294',    # Xiaomi Redmi Note 13 Pro+ a S/ 1.250.000
    'MPE959377798',    # Lenovo V15 a S/ 29.000
    'MPE1090653346',   # Televisor Daewoo a S/ 99.999
    'MPE1094203286',   # Televisor Daewoo a S/ 10
    'MPE1094202682',   # Televisor Daewoo a S/ 10
    'MPE1090833326',   # Refrigerador Daewoo a S/ 999.999
    'MPE1092274734',   # Refrigeradora Indurama a S/ 2
]

marca_excluidos = unicas['id_publicacion'].isin(IDS_EXCLUIDOS)
excluidos = unicas.loc[marca_excluidos].assign(motivo_exclusion=MOTIVO)
assert len(excluidos) == len(IDS_EXCLUIDOS), 'Algún identificador excluido no está en el dataset.'
excluidos.to_csv(CALIDAD / 'precios_excluidos.csv', index=False, encoding='utf-8')
unicas = unicas.loc[~marca_excluidos]

# Conservamos atributos con cobertura suficiente. Los textos crudos quedan en el CSV original.
columnas = ['id_publicacion', 'id_extraido', 'categoria', 'titulo', 'moneda',
            'precio_actual', 'numero_cuotas', 'monto_cuota', 'cuotas_texto',
            'pagina', 'posicion_pagina', 'url_publicacion', 'url_fuente', 'fecha_extraccion']
limpio = unicas[columnas].reset_index(drop=True)
calidad = pd.DataFrame({'variable': limpio.columns,
    'tipo_pandas': [str(limpio[c].dtype) for c in limpio],
    'nulos': [int(limpio[c].isna().sum()) for c in limpio],
    'porcentaje_nulos': [limpio[c].isna().mean() * 100 for c in limpio],
    'valores_distintos': [limpio[c].nunique() for c in limpio]})
print('Filas originales:', len(raw))
print('Precios excluidos:', len(excluidos))
print('Dimensiones finales:', limpio.shape)
print(calidad.to_string(index=False))

Filas originales: 5080
Precios excluidos: 7
Dimensiones finales: (4412, 14)
        variable    tipo_pandas  nulos  porcentaje_nulos  valores_distintos
  id_publicacion         string      0            0.0000               4412
     id_extraido         string      0            0.0000               4412
       categoria         string      0            0.0000                  5
          titulo         string      0            0.0000               4181
          moneda         string      0            0.0000                  1
   precio_actual        Float64      0            0.0000               1952
   numero_cuotas          Int64     55            1.2466                  2
     monto_cuota        Float64     55            1.2466               2225
    cuotas_texto         string     55            1.2466               2335
          pagina          Int64      0            0.0000                 20
 posicion_pagina          Int64      0            0.0000                 60
 url_publica

## 2.5 Verificación y exportación
El dataset principal mantiene más de seis atributos, variables numéricas y categóricas, al menos mil ofertas identificadas y como máximo 10 % de nulos por columna. Las columnas de página y posición son metadatos; no se presentan como características comerciales del producto.

In [13]:
assert len(limpio) >= 1000, 'No se alcanza el mínimo de registros.'
assert len(limpio.columns) >= 6
assert limpio.id_publicacion.is_unique
assert calidad.porcentaje_nulos.le(10).all(), 'Hay columnas que superan el 10 % de nulos.'
assert limpio.precio_actual.gt(0).all()
assert limpio.numero_cuotas.dropna().gt(0).all()
assert limpio.monto_cuota.dropna().gt(0).all()
assert len(raw) == len(revision) + int(repetidas.sum()) + len(excluidos) + len(limpio)
limpio.to_csv(LIMPIOS / 'mercadolibre_limpio.csv', index=False, encoding='utf-8', date_format='%Y-%m-%d %H:%M:%S')
calidad.to_csv(CALIDAD / 'calidad_por_columna.csv', index=False, encoding='utf-8')
releido = pd.read_csv(LIMPIOS / 'mercadolibre_limpio.csv', encoding='utf-8')
assert releido.shape == limpio.shape
assert releido.id_publicacion.is_unique
assert releido.isna().mean().le(.10).all()
resumen = limpio.groupby('categoria').agg(registros=('id_publicacion', 'size'),
    precio_minimo=('precio_actual', 'min'), precio_mediano=('precio_actual', 'median'),
    precio_promedio=('precio_actual', 'mean'), precio_maximo=('precio_actual', 'max')).round(2)
resumen.to_csv(CALIDAD / 'resumen_por_categoria.csv', encoding='utf-8')
print(resumen.to_string())
print('Validaciones aprobadas y CSV reabierto correctamente.')

                registros  precio_minimo  precio_mediano  precio_promedio  precio_maximo
categoria                                                                               
audifonos             959           16.9           138.0           376.08         8959.0
celulares             937          38.61          1632.0          2075.04         9498.0
laptops               928          459.0          4898.5          5807.42        24665.0
refrigeradoras        646           52.0          1314.0          1805.08        20741.0
televisores           942           25.0          1289.5          1907.65        59999.0
Validaciones aprobadas y CSV reabierto correctamente.


## 2.6 Texto de la sección Dataset preliminar
La siguiente celda escribe `documentos/Dataset_preliminar.md` con los resultados reales de esta ejecución: es la sección 4 del informe. `documentos/notas_integracion.md` registra qué deben ajustar las otras partes.

In [14]:
def tabla(frame):
    encabezado = '| ' + ' | '.join(map(str, frame.columns)) + ' |'
    separador = '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |'
    filas = ['| ' + ' | '.join(str(x).replace('|', '/') for x in fila) + ' |' for fila in frame.itertuples(index=False, name=None)]
    return '\n'.join([encabezado, separador] + filas)

texto = f"""# 4. Dataset preliminar

## 4.1 Descripción de la muestra
Se procesaron {len(raw):,} observaciones crudas de Mercado Libre Perú, obtenidas entre {df.fecha_extraccion.min()} y {df.fecha_extraccion.max()}, según las marcas temporales del scraper. El archivo no registra zona horaria. Las búsquedas de origen fueron celulares, laptops, televisores, refrigeradoras y audifonos. El dataset final contiene {len(limpio):,} ofertas con identificadores únicos y {len(limpio.columns)} columnas.

La unidad de observación es la oferta identificada en la URL, conservando su última aparición en el archivo. La muestra procede de páginas de resultados y no es aleatoria ni representativa de todo el mercado peruano. La categoría es la búsqueda de origen y puede incluir accesorios o productos relacionados. Los precios describen ofertas, no transacciones realizadas.

## 4.2 Preparación y control de calidad
Se normalizaron espacios, se transformaron precios a formato numérico, se estandarizó la moneda a PEN y se convirtieron las fechas a datetime. Se extrajeron el número de cuotas y el monto por cuota de los textos observados, manteniendo los nulos cuando la información no estaba disponible.

El identificador extraído originalmente por el scraper se conserva como id_extraido. Para id_publicacion se tomó el parámetro wid de la URL o, cuando estaba ausente, el código de una ruta directa sin /p/. Se separaron {len(revision)} filas sin campos esenciales válidos y se retiraron {int(repetidas.sum())} observaciones repetidas, conservando la última por fecha y orden de archivo. Adicionalmente se excluyeron {len(excluidos)} ofertas con precios cuestionados, identificadas por código y precio: Xiaomi Redmi Note 13 Pro+ a S/ 1.250.000, refrigerador Daewoo a S/ 999.999, televisor Daewoo a S/ 99.999, refrigeradora Indurama a S/ 2, dos televisores Daewoo a S/ 10 y Lenovo V15 a S/ 29.000. Son exclusiones puntuales acordadas para esta muestra, no límites generales aplicables a todos los productos. Estos importes ya estaban en el CSV crudo; no fueron generados por la conversión numérica. La exclusión es una decisión de calidad para el análisis, sin verificar el precio real ni imputar reemplazos. Los registros separados se guardaron como evidencia para revisión.

No se encontraron duplicados de fila completa en el CSV original ({int(raw.duplicated().sum())}); las repeticiones se detectaron por identificador de oferta, aunque sus posiciones, fechas o enlaces de seguimiento fueran diferentes.

Se excluyeron precio_anterior_texto y vendedor_texto del dataset principal debido a sus ausencias originales de {raw.precio_anterior_texto.isna().mean()*100:.2f} % y {raw.vendedor_texto.isna().mean()*100:.2f} %, respectivamente. Ambas columnas permanecen en el CSV crudo. No se imputaron precios anteriores, vendedores ni descuentos. La falta de un precio anterior no permite concluir que el descuento sea cero.

## 4.3 Variables y cobertura
Las variables comerciales incluyen titulo, categoria, moneda, precio_actual, numero_cuotas, monto_cuota y cuotas_texto. Los identificadores, páginas, posiciones, URL y fecha proporcionan trazabilidad. El dataset incluye variables numéricas y categóricas, supera los 1000 registros y los seis atributos mínimos, y ninguna columna supera el 10 % de valores nulos. La tabla presenta los tipos en pandas antes de exportar; CSV no conserva tipos y deben declararse al volver a cargarlo.

{tabla(calidad.round(2))}

## 4.4 Estadísticos descriptivos
Todos los precios se expresan en soles (PEN). Tras las exclusiones puntuales descritas, se conservaron los demás valores extremos porque su magnitud por sí sola no prueba un error. Las estadísticas siguientes excluyen los siete casos documentados en precios_excluidos.csv. No se ha verificado el precio real de cada publicación ni se afirma que todos los precios restantes sean correctos. Las diferencias entre categorías no equivalen a diferencias entre productos comparables.

{tabla(resumen.reset_index())}

## 4.5 Vista previa
Se muestra una selección de columnas de las primeras cinco filas. El CSV contiene la totalidad de los atributos descritos.

{tabla(limpio[['id_publicacion', 'categoria', 'titulo', 'precio_actual', 'numero_cuotas']].head())}

## 4.6 Limitaciones
No se extrajeron calificación, número de opiniones, condición, marca ni envío gratuito. Por tanto, estos datos no permiten analizar esas características ni la reputación del vendedor. Tampoco permiten calcular descuentos de manera completa. La deduplicación conserva una sola observación por oferta y no constituye una serie histórica. La identidad se recuperó de las URL guardadas y no se verificó mediante consultas adicionales al sitio.
"""
(DOCUMENTOS / 'Dataset_preliminar.md').write_text(texto, encoding='utf-8')
print('Sección 4 del informe guardada en', DOCUMENTOS / 'Dataset_preliminar.md')

Sección 4 del informe guardada en c:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\documentos\Dataset_preliminar.md


# 3. Resultado

El dataset limpio y las tablas de control quedan versionados en el repositorio y
`documentos/Dataset_preliminar.md` se copia a la sección 4 del informe. `filas_para_revision.csv`,
`duplicados_retirados.csv` y `precios_excluidos.csv` son trazabilidad de las decisiones de limpieza: no
deben volver a mezclarse con el dataset limpio.

Entregables del enunciado que salen de este notebook: el dataset preliminar en CSV UTF-8
(`datos/limpios/mercadolibre_limpio.csv`) y el texto de la sección Dataset preliminar del informe. El
diccionario de datos está en `documentos/`.